In [8]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# external data
import holidays
import yfinance as yf
import requests


In [4]:
# connect database
import os
if os.path.exists('pass.env'):
    load_dotenv('pass.env')
else:
    load_dotenv('../pass.env')


db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

db_url = f"postgresql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

In [5]:
# query start to end date
date_query = """
    SELECT 
        MIN(DATE(order_purchase_timestamp)) AS min_date,
        MAX(DATE(order_purchase_timestamp)) AS max_date
    FROM 
        orders;
"""
with engine.connect() as connection:
    result = connection.execute(text(date_query)).fetchone()
    
    # Extract dates and convert to string format YYYY-MM-DD
    start_date = result.min_date.strftime('%Y-%m-%d')
    end_date = result.max_date.strftime('%Y-%m-%d')

print(f"Data start date: {start_date}")
print(f"Data end date: {end_date}")

dates = pd.date_range(start=start_date, end=end_date)
dim_date = pd.DataFrame({'full_date': dates})

Data start date: 2016-09-04
Data end date: 2018-10-17


In [ ]:
# add brazill holidays
br_holidays = holidays.Brazil(years=[2016, 2017, 2018])

def get_holiday_name(dt):
    # Return holiday name if exists, else None
    return br_holidays.get(dt)

dim_date['holiday_name'] = dim_date['full_date'].apply(get_holiday_name)
dim_date['is_holiday'] = dim_date['holiday_name'].notnull().astype(int)

# add exchange rate (USD to BRL)
brl_df = yf.download('BRL=X', start=start_date, end='2018-11-01')

# Extract only the 'Close' series to completely bypass the MultiIndex issue
brl_series = brl_df['Close']
brl_data = brl_series.reset_index()
brl_data.columns = ['full_date', 'usd_brl_rate']

# Standardize datetime format by removing timezone information
brl_data['full_date'] = pd.to_datetime(brl_data['full_date']).dt.tz_localize(None)
dim_date['full_date'] = pd.to_datetime(dim_date['full_date']).dt.tz_localize(None)

dim_date = dim_date.merge(brl_data, on='full_date', how='left')

# forward fill missing exchange rates (e.g., weekends)
dim_date['usd_brl_rate'] = dim_date['usd_brl_rate'].ffill()

# Write to database
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)

[*********************100%***********************]  1 of 1 completed


774

In [ ]:
# SQL Query using Window Functions (80% Cumulative Volume)
pareto_query = """
    WITH StateVolume AS (
        SELECT 
            c.customer_state,
            COUNT(DISTINCT o.order_id) AS order_count
        FROM 
            customers c
        JOIN 
            orders o ON c.customer_id = o.customer_id
        GROUP BY 
            c.customer_state
    ),
    StateRunningTotal AS (
        SELECT 
            customer_state,
            order_count,
            SUM(order_count) OVER (ORDER BY order_count DESC) AS cumulative_orders,
            SUM(order_count) OVER () AS grand_total_orders
        FROM 
            StateVolume
    )
    SELECT 
        customer_state,
        order_count,
        cumulative_orders,
        grand_total_orders,
        ROUND((cumulative_orders * 100.0 / grand_total_orders), 2) AS cumulative_percentage
    FROM 
        StateRunningTotal
    ORDER BY 
        order_count DESC;
"""

# Execute and Analyze
with engine.connect() as connection:
    df_pareto = pd.read_sql(text(pareto_query), connection)

# Filter states that cover approximately 80% of total orders
threshold_percentage = 80.0

# Identify the exact cutoff index where cumulative percentage hits or just crosses 80%
# use < threshold and then add 1 to ensure we include the state that crosses the line
target_states_df = df_pareto[df_pareto['cumulative_percentage'] - (df_pareto['order_count'] * 100 / df_pareto['grand_total_orders']) <= threshold_percentage]

target_states_list = target_states_df['customer_state'].tolist()
actual_coverage = target_states_df['cumulative_percentage'].max()

# the results for validation
print("Pareto Analysis: States covering ~80% of total orders")
print("-" * 60)
print(target_states_df[['customer_state', 'order_count', 'cumulative_percentage']].to_string(index=False))
print("-" * 60)
print(f"Total states required: {len(target_states_list)}")
print(f"Actual volume coverage: {actual_coverage}%")
print(f"Target States List for Weather API: {target_states_list}")

Pareto Analysis: States covering ~80% of total orders
------------------------------------------------------------
customer_state  order_count  cumulative_percentage
            SP        41746                  41.98
            RJ        12852                  54.90
            MG        11635                  66.61
            RS         5466                  72.10
            PR         5045                  77.18
            SC         3637                  80.83
------------------------------------------------------------
Total states required: 6
Actual volume coverage: 80.83%
Target States List for Weather API: ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC']


In [10]:
# Load configuration and connect to database
import os
if os.path.exists('pass.env'):
    load_dotenv('pass.env')
else:
    load_dotenv('../pass.env')

db_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

target_states = ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC']

print("Starting Weather Data Pipeline using direct Open-Meteo API")

with engine.connect() as connection:
    # Get dynamic date range from actual orders
    date_query = """
        SELECT 
            MIN(DATE(order_purchase_timestamp)) AS min_date,
            MAX(DATE(order_purchase_timestamp)) AS max_date
        FROM orders;
    """
    date_result = connection.execute(text(date_query)).fetchone()
    
    # Format dates as strings for the API request
    start_date_str = date_result.min_date.strftime('%Y-%m-%d')
    end_date_str = date_result.max_date.strftime('%Y-%m-%d')
    
    print(f"Timeframe: {start_date_str} to {end_date_str}")

    weather_data_list = []
    
    # Process each state directly using Geolocation and API Requests
    for state in target_states:
        print(f"Processing state: {state}...")
        
        geo_query = f"""
            SELECT 
                AVG(geolocation_lat) as avg_lat, 
                AVG(geolocation_lng) as avg_lng
            FROM geolocation
            WHERE geolocation_state = '{state}';
        """
        geo_result = connection.execute(text(geo_query)).fetchone()
        
        if geo_result and geo_result.avg_lat is not None:
            lat = round(float(geo_result.avg_lat), 4)
            lon = round(float(geo_result.avg_lng), 4)
            
            api_url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date_str}&end_date={end_date_str}&daily=temperature_2m_mean,precipitation_sum&timezone=America%2FSao_Paulo"
            
            try:
                response = requests.get(api_url)
                response.raise_for_status()
                data = response.json()
                
                # Check if daily data exists in response
                if "daily" in data:
                    state_weather = pd.DataFrame({
                        'weather_date': data["daily"]["time"],
                        'avg_temp_celsius': data["daily"]["temperature_2m_mean"],
                        'precipitation_mm': data["daily"]["precipitation_sum"]
                    })
                    state_weather['state_code'] = state
                    weather_data_list.append(state_weather)
                    print(f" - Success: Downloaded weather data for {state} (Lat: {lat}, Lon: {lon})")
                else:
                    print(f" - Warning: API returned empty daily data for {state}")
                    
            except Exception as e:
                print(f" - Error fetching data for {state}: {e}")

# Transform and Load to PostgreSQL
if weather_data_list:
    print("Transforming and loading data into PostgreSQL...")
    dim_weather = pd.concat(weather_data_list, ignore_index=True)
    
    # Convert date string to datetime and remove timezone information
    dim_weather['weather_date'] = pd.to_datetime(dim_weather['weather_date']).dt.tz_localize(None)
    
    # Forward fill missing temperature and fill missing precipitation with 0
    dim_weather['avg_temp_celsius'] = dim_weather['avg_temp_celsius'].ffill()
    dim_weather['precipitation_mm'] = dim_weather['precipitation_mm'].fillna(0)
    
    # Write to database
    dim_weather.to_sql('dim_weather', engine, if_exists='replace', index=False)
    
    # Add primary key constraint using raw SQL
    with engine.connect() as connection:
        connection.execute(text("ALTER TABLE dim_weather ADD PRIMARY KEY (weather_date, state_code);"))
        connection.commit()
        
    print(f"Success! Inserted {len(dim_weather)} weather records into 'dim_weather' table.")
else:
    print("Error: No weather data fetched from API.")

Starting Weather Data Pipeline using direct Open-Meteo API
Timeframe: 2016-09-04 to 2018-10-17
Processing state: SP...
 - Success: Downloaded weather data for SP (Lat: -23.1553, Lon: -47.0841)
Processing state: RJ...
 - Success: Downloaded weather data for RJ (Lat: -22.7435, Lon: -43.1555)
Processing state: MG...
 - Success: Downloaded weather data for MG (Lat: -19.8646, Lon: -44.4216)
Processing state: RS...
 - Success: Downloaded weather data for RS (Lat: -29.6792, Lon: -52.0327)
Processing state: PR...
 - Success: Downloaded weather data for PR (Lat: -24.7936, Lon: -50.8797)
Processing state: SC...
 - Success: Downloaded weather data for SC (Lat: -27.2225, Lon: -49.6179)
Transforming and loading data into PostgreSQL...
Success! Inserted 4644 weather records into 'dim_weather' table.
